# Multi-Turn conversations

```
Bedrock and Claude do not store any messages. None of the messages you send get stored, and none of the responses you receive are stored either. Each API call is completely independent.
```

### No Message Storage

Bedrock and Claude do not store any messages. None of the messages you send get stored, and none of the responses you receive are stored either. Each API call is completely independent.

To have a conversation with multiple messages that maintain context, you need to:

    Manually maintain a list of all messages in your code
    Provide that entire list of messages with each follow-up request



In [1]:
# Importing Boto3
import boto3
import json

In [2]:
# Creating an client of bedrock
claude_bedrock = boto3.client('bedrock-runtime',  region_name = 'us-west-2')
model_id = 'global.anthropic.claude-sonnet-4-6'
system_prompts = [{"text": "You are Delhi Math Radio Jockey"}]
message_1 = {'role': 'user', 'content': [{'text': 'What is 5 * 5?.'}]}

In [3]:
# Calling Invoke Model function
response = claude_bedrock.converse(
    modelId = model_id,
    messages = [message_1],
    system = system_prompts
    )

In [4]:
response

{'ResponseMetadata': {'RequestId': '62791aa2-0188-4db9-8cb6-31722de81c9e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 03 Mar 2026 05:10:52 GMT',
   'content-type': 'application/json',
   'content-length': '988',
   'connection': 'keep-alive',
   'x-amzn-requestid': '62791aa2-0188-4db9-8cb6-31722de81c9e'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': '🎵 *Delhi Math Radio jingle plays* 🎵\n\nAye aye aye! Namaskar dosto! Yahan se bol raha hai aapka favorite **Delhi Math Radio**! 📻\n\nAb suniye, aaj ka **superhit sawal** - 5 guna 5! 🔥\n\nToh dosto, jab hum **5 ka 5 baar addition** karte hain...\n\n5 + 5 + 5 + 5 + 5 = **25!** 🎉\n\nHaan haan haan! **PACHEES!** 25! ✨\n\nJaise Delhi mein **25 rupaye** mein ek cutting chai milti hai - **ekdum perfect!** ☕😄\n\nToh yaad rakho dosto -\n🎵 *"Paanch guna paanch, hoga PACHEES,*\n*Math seekhlo, life mein milegi breeze!"* 🎵\n\nYeh tha aapka **Delhi Math Radio** - jahan numbers bhi bolte hain *d

In [5]:
response['output']['message']['content'][0]['text']

'🎵 *Delhi Math Radio jingle plays* 🎵\n\nAye aye aye! Namaskar dosto! Yahan se bol raha hai aapka favorite **Delhi Math Radio**! 📻\n\nAb suniye, aaj ka **superhit sawal** - 5 guna 5! 🔥\n\nToh dosto, jab hum **5 ka 5 baar addition** karte hain...\n\n5 + 5 + 5 + 5 + 5 = **25!** 🎉\n\nHaan haan haan! **PACHEES!** 25! ✨\n\nJaise Delhi mein **25 rupaye** mein ek cutting chai milti hai - **ekdum perfect!** ☕😄\n\nToh yaad rakho dosto -\n🎵 *"Paanch guna paanch, hoga PACHEES,*\n*Math seekhlo, life mein milegi breeze!"* 🎵\n\nYeh tha aapka **Delhi Math Radio** - jahan numbers bhi bolte hain *dil ki baat!* ❤️\n\nStay tuned, stay sharp! 📻✌️'

## If you see, the model is not able to recall last exchange of messages

In [6]:
message_1 = {'role': 'user', 'content': [{'text': 'now add 10 to that?'}]}

In [7]:
# Calling Invoke Model function
response = claude_bedrock.converse(
    modelId = model_id,
    messages = [message_1],
    )

In [8]:
response['output']['message']['content'][0]['text']

"I don't have context for what number you're referring to. This appears to be the start of our conversation, so I'm not sure what number you'd like me to add 10 to.\n\nCould you provide the number you'd like me to work with? 😊"

### Why Context Matters

Let's see what happens without proper context. If you send just "And 3 more?" as a standalone message, Claude has no idea what you're referring to. It will do its best to respond, but the answer won't make sense because it lacks the context of your previous conversation.

When you send only the follow-up question, Claude sees just that isolated message and tries to respond without knowing about the previous "What's 1+1?" exchange.

### Building Conversation Context

To maintain context, you need to include the full conversation history in each request. Here's how it works:

Your message list should contain all previous exchanges - both user messages and assistant responses. When you send this complete context, Claude can understand that "And 3 more?" refers to adding 3 to the previous result of 2.

### Helper Functions for Message Management

To make conversation management easier, you can create helper functions:

```python
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": [
            {"text": text}
        ]
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant", 
        "content": [
            {"text": text}
        ]
    }
    messages.append(assistant_message)

def chat(messages):
    response = client.converse(
        modelId=model_id,
        messages=messages
    )
    return response["output"]["message"]["content"][0]["text"]
```


### Helper Function for Message Management

In [9]:
# We are creating a function whose job is to add "USER" Message to the list

def add_user_message(messages, text):
    user_message = {
        "role" : "user",
        "content" : [
            { "text" : text }
        ]
    }
    messages.append(user_message)


# Function to add "Assistant" messages to the list 
def add_assistant_message(messages, text):
    assistant_message = {
        "role" : "assistant",
        "content" : [
            { "text" : text }
        ]
    }
    messages.append(assistant_message)


# function to invoke the model and return the assistant response
def chat(messages):

    response = claude_bedrock.converse(
        modelId = model_id,
        messages = messages
    )

    return response['output']['message']['content'][0]['text']

### Implementing Multi-Turn Conversations

In [11]:
# Making an empty list where we will store the messages
messages = []

# Add in the initial user question of "What 2+3?"
add_user_message(messages, "what's 2+3?")

#Pass the list of messages into the chat to get an answer
answer = chat(messages)  # response from chat funtion defined earlier will be saved to answer variable - this is assistant message

# Take the answer and add it to the list - answer = assistant message
add_assistant_message(messages, answer)

# Adding the follow up question by user
add_user_message(messages, "What would be the answer if I add 5 to it?")

# Response from model - assistant message
answer = chat(messages)

answer

'If you add 5 to the previous answer of 5, you get:\n\n5 + 5 = **10**'

### Implementing Multi-Turn Conversations

Here's how to build a conversation step by step:

```python
# Make a starting list of messages
messages = []

# Add in the initial user question of "What's 1+1?"
add_user_message(messages, "What's 1+1?")

# Pass the list of messages into chat to get an answer
answer = chat(messages)

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

# Add in the user's followup question
add_user_message(messages, "And 3 more added to that?")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)
print(answer)

This approach ensures Claude has the full context and can respond appropriately: "Starting with the result of 1+1 = 2, if we add 3 more to that, we get: 2 + 3 = 5"
```

### Message Role Alternation

When building your message list, always ensure that message roles alternate properly:

Your conversation should follow the pattern: user → assistant → user → assistant. Never have two user messages in a row or two assistant messages in a row. This alternating pattern is required by the API and reflects natural conversation flow.

While this manual message management might seem tedious at first, you'll quickly get used to it. This pattern is fundamental to building any application that needs to maintain conversational context with Claude.
